In [4]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats

import dask.dataframe as dd
from pathlib import Path
import glob

import datetime as dt

import matplotlib.pyplot as plt
from matplotlib import colors
import soundfile as sf
import matplotlib.patches as patches

In [5]:
import sys

sys.path.append("../src")
sys.path.append("../src/activity")

In [6]:
import subsampling as ss
import activity.activity_assembly as actvt
from core import SITE_NAMES, FREQ_GROUPS

from cli import get_file_paths
import plot
import pipeline

In [7]:
import suncalc

import activity.activity_assembly as actvt

from core import DC_COLOR_MAPPINGS, SEATTLE_LATITUDE, SEATTLE_LONGITUDE

In [8]:
avail = np.arange(0, 720, 6) + 6
reset_24 = avail[np.where((24*60 % avail) == 0)[0]]
reset_24

array([  6,  12,  18,  24,  30,  36,  48,  60,  72,  90,  96, 120, 144,
       180, 240, 288, 360, 480, 720])

In [9]:
cycle_lengths = [10]
percent_ons = [1/2]
specific_dc_tag = "30of30"

data_params = dict()
data_params["year"] = '2022'
data_params["cycle_lengths"] = cycle_lengths
data_params["percent_ons"] = percent_ons
dc_tags = ss.get_list_of_dc_tags(data_params["cycle_lengths"], data_params["percent_ons"])
data_params["dc_tags"] = dc_tags
data_params["cur_dc_tag"] = specific_dc_tag
data_params['detector_tag'] = 'bd2'
data_params['bin_size'] = '30'
data_params['det_prob_threshold'] = 0.35
data_params['recording_start'] = '00:00'
data_params['recording_end'] = '16:00'
data_params['assembly_type'] = 'kmeans'

pipeline_params = dict()
pipeline_params['assemble_location_summary'] = True
pipeline_params["read_csv"] = False
pipeline_params["save_activity_grid"] = False
pipeline_params["save_presence_grid"] = False
pipeline_params["save_dc_night_comparisons"] = False
pipeline_params["save_activity_dc_comparisons"] = True
pipeline_params["save_presence_dc_comparisons"] = True
pipeline_params["show_plots"] = True
pipeline_params["show_PST"] = True

In [10]:
site_key = 'Carp'
type_key = ''
data_params["site_name"] = SITE_NAMES[site_key]
data_params["site_tag"] = site_key
data_params["type_tag"] = type_key

file_paths = get_file_paths(data_params)

In [11]:
# location_thresh_df = pipeline.prepare_location_sumary(data_params, pipeline_params, file_paths)
init_location_sum = actvt.assemble_initial_location_summary(file_paths) 
init_location_sum.reset_index(inplace=True)
init_location_sum.rename({'index':'index_in_file'}, axis='columns', inplace=True)
init_location_sum

,index_in_file,ref_time,call_start_time,call_end_time,start_time,end_time,low_freq,high_freq,event,class,class_prob,det_prob,individual,input_file,Site name,Recover Folder,SD Card,File Duration
0,0,2022-07-13 00:00:51.839500,2022-07-13 00:00:51.839500,2022-07-13 00:00:51.854300,51.8395,51.8543,22890.0,27138.0,Echolocation,Nyctalus leisleri,0.096,0.227,-1,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Carp Pond,recover-20220715,8,1795
1,1,2022-07-13 00:01:17.840500,2022-07-13 00:01:17.840500,2022-07-13 00:01:17.855800,77.8405,77.8558,22890.0,26911.0,Echolocation,Nyctalus leisleri,0.095,0.204,-1,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Carp Pond,recover-20220715,8,1795
2,2,2022-07-13 00:03:34.287500,2022-07-13 00:03:34.287500,2022-07-13 00:03:34.298800,214.2875,214.2988,32343.0,37435.0,Echolocation,Pipistrellus nathusii,0.162,0.355,-1,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Carp Pond,recover-20220715,8,1795
3,3,2022-07-13 00:04:20.330500,2022-07-13 00:04:20.330500,2022-07-13 00:04:20.343800,260.3305,260.3438,24609.0,29278.0,Echolocation,Nyctalus leisleri,0.133,0.227,-1,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Carp Pond,recover-20220715,8,1795
4,4,2022-07-13 00:04:45.531500,2022-07-13 00:04:45.531500,2022-07-13 00:04:45.539600,285.5315,285.5396,43515.0,47936.0,Echolocation,Pipistrellus pipistrellus,0.205,0.216,-1,/mnt/ubna_data_01_mir/recover-20220715/UBNA_00...,Carp Pond,recover-20220715,8,1795
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2430081,2,2022-10-17 15:37:12.995500,2022-10-17 15:37:12.995500,2022-10-17 15:37:13.011500,432.9955,433.0115,22031.0,27095.0,Echolocation,Nyctalus noctula,0.098,0.207,-1,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Carp Pond,recover-20221017,10,1795
2430082,3,2022-10-17 15:42:37.264500,2022-10-17 15:42:37.264500,2022-10-17 15:42:37.280400,757.2645,757.2804,11718.0,17316.0,Echolocation,Nyctalus leisleri,0.138,0.203,-1,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Carp Pond,recover-20221017,10,1795
2430083,4,2022-10-17 15:45:42.987500,2022-10-17 15:45:42.987500,2022-10-17 15:45:42.993700,942.9875,942.9937,22890.0,34055.0,Echolocation,Plecotus austriacus,0.106,0.201,-1,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Carp Pond,recover-20221017,10,1795
2430084,5,2022-10-17 15:49:20.239500,2022-10-17 15:49:20.239500,2022-10-17 15:49:20.257500,1160.2395,1160.2575,21171.0,24995.0,Echolocation,Nyctalus noctula,0.099,0.204,-1,/mnt/ubna_data_02_mir/recover-20221017/UBNA_01...,Carp Pond,recover-20221017,10,1795


In [12]:
def add_frequency_group_to_file_dets(file_dets, location_classes):
    file_classes = location_classes[pd.to_datetime(location_classes['file_name'], 
                                                   format='%Y%m%d_%H%M%S.WAV', exact=False)==file_dets.name].copy()

    file_dets.insert(0, 'index_in_summary', file_dets.index)
    file_dets.set_index('index_in_file', inplace=True)

    classified = file_classes['KMEANS_CLASSES']!=''
    file_classes.loc[classified, 'peak_frequency'] = file_classes.loc[classified, 'peak_frequency'].astype('float64')
    file_classes.loc[classified, 'SNR'] = file_classes.loc[classified, 'SNR'].astype('float64')

    file_dets.insert(0, 'peak_frequency', [np.NaN]*len(file_dets))
    file_dets.insert(0, 'SNR', [np.NaN]*len(file_dets))
    file_dets.loc[file_classes['index_in_file'], 'freq_group'] = file_classes['KMEANS_CLASSES'].values
    file_dets.loc[file_classes['index_in_file'], 'peak_frequency'] = file_classes['peak_frequency'].values
    file_dets.loc[file_classes['index_in_file'], 'SNR'] = file_classes['SNR'].values

    for group in ['LF', 'HF']:
        group_classified_dets = (file_dets['freq_group']==group)

        low_assert1 = (file_dets.loc[group_classified_dets, 'peak_frequency'] > (file_dets.loc[group_classified_dets, 'low_freq']).median()-4000)
        low_assert2 = (file_dets.loc[group_classified_dets, 'peak_frequency'] > (file_dets.loc[group_classified_dets, 'low_freq'])-4000)
        assert(low_assert1|low_assert2).all()
        high_assert1 = (file_dets.loc[group_classified_dets, 'peak_frequency'] < (file_dets.loc[group_classified_dets, 'high_freq']).median()+4000)
        high_assert2 = (file_dets.loc[group_classified_dets, 'peak_frequency'] < (file_dets.loc[group_classified_dets, 'high_freq'])+4000)
        assert(high_assert1|high_assert2).all()

    return file_dets

def add_frequency_groups_to_summary_using_kmeans(location_df, file_paths, data_params, save=True):
    location_df.insert(0, 'freq_group', '')
    location_classes = pd.read_csv(Path(file_paths['SITE_classes_file']), index_col=0)
    location_df.insert(0, 'input_file_dt', pd.to_datetime(location_df['input_file'], format='%Y%m%d_%H%M%S.WAV', exact=False))
    location_df_grouped = location_df.groupby('input_file_dt', group_keys=True)

    location_df_classified = location_df_grouped.apply(lambda x: add_frequency_group_to_file_dets(x, location_classes))

    location_df_only_classified = location_df_classified.loc[location_df_classified['freq_group']!='']
    location_df_only_classified = location_df_only_classified.droplevel(level=0)
    location_df_only_classified = location_df_only_classified.reset_index()

    if data_params['type_tag'] != '':
        location_df_only_classified = location_df_only_classified.loc[location_df_only_classified['freq_group']==data_params['type_tag']]

    if save:
        location_df_only_classified.to_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv')

    return location_df_only_classified

In [13]:
# location_sum = add_frequency_groups_to_summary_using_kmeans(init_location_sum.copy(), file_paths, data_params)
location_df = init_location_sum.copy()
location_df.insert(0, 'freq_group', '')
location_classes = pd.read_csv(Path(file_paths['SITE_classes_file']), index_col=0)
location_df.insert(0, 'input_file_dt', pd.to_datetime(location_df['input_file'], format='%Y%m%d_%H%M%S.WAV', exact=False))
location_df_grouped = location_df.groupby('input_file_dt', group_keys=True)
location_classes

,index,KMEANS_CLASSES,peak_frequency,SNR,index_in_file,index_in_summary,file_name,sampling_rate
0,0,LF,29760.0,19.337273,0,0,20220713_043000.WAV,250000
1,1,LF,24960.0,7.658908,1,1,20220713_043000.WAV,250000
2,2,LF,24960.0,5.029702,2,2,20220713_043000.WAV,250000
3,3,LF,26880.0,8.310447,3,3,20220713_043000.WAV,250000
4,4,LF,24960.0,9.005425,4,4,20220713_043000.WAV,250000
...,...,...,...,...,...,...,...,...
1012474,71,LF,25920.0,8.130118,72,1016410,20221017_133000.WAV,192000
1012475,72,LF,24960.0,7.821952,73,1016411,20221017_133000.WAV,192000
1012476,73,LF,25920.0,4.395280,74,1016412,20221017_133000.WAV,192000
1012477,74,LF,24960.0,11.030292,75,1016413,20221017_133000.WAV,192000


In [14]:
import re

def relabel_drivenames_to_mirrors(filepaths):
    drivename = re.compile(r'ubna_data_0[0-9]/')
    for i, fp in enumerate(filepaths):
        if bool(drivename.search(fp)):
            d_name = drivename.search(fp).group()
            replace_d_name = f'{d_name[:-1]}_mir/'
            filepaths[i] = filepaths[i].replace(d_name, replace_d_name)

    return filepaths


def get_params_relevant_to_data_at_location(cfg):
    data_params = dict()
    data_params["type_tag"] = ''
    data_params["cur_dc_tag"] = "30of30"
    data_params["site_tag"] = cfg['site']
    data_params['site_name'] = SITE_NAMES[cfg['site']]
    data_params['assembly_type'] = 'thresh'
    data_params['detector_tag'] = cfg['detector']
    print(f"Searching for files from {data_params['site_name']}")

    file_paths = get_file_paths(data_params)
    location_sum_df = pd.read_csv(f'{file_paths["SITE_folder"]}/{file_paths["detector_TYPE_SITE_YEAR"]}.csv', low_memory=False, index_col=0)
    location_sum_df.reset_index(inplace=True)
    location_sum_df.rename({'index':'index_in_summary'}, axis='columns', inplace=True)

    data_params['good_audio_files'] = location_sum_df['input_file'].copy().unique()
    print(f"Will be looking at {len(data_params['good_audio_files'])} files from {data_params['site_name']}")

    return location_sum_df, data_params


In [15]:
cfg= dict()
cfg['site'] = 'Carp'
cfg['recording_start'] = '00:00'
cfg['recording_end'] = '16:00'
cfg['detector'] = 'bd2'

In [16]:
corrected_classifications = pd.DataFrame()
classifications = pd.DataFrame()
location_sum_df, data_params = get_params_relevant_to_data_at_location(cfg)
file_raw_title = f'2022_{cfg["detector"]}{data_params["site_tag"]}_call_classes_raw'
file_corrected_title = f'2022_{cfg["detector"]}{data_params["site_tag"]}_call_classes'

Searching for files from Carp Pond
Will be looking at 2843 files from Carp Pond


In [19]:
input_file = data_params['good_audio_files'][0]
input_file

'/mnt/ubna_data_01_mir/recover-20220715/UBNA_008/20220713_000000.WAV'

In [20]:
file_path = '/'.join(Path(input_file).parts[2:])
cleaned_path = re.sub(r"(ubna_data_\d+)_mir", r"\1", file_path)
osn_file_path = Path(f'bio230143-bucket01/{cleaned_path}')
osn_file_path

PosixPath('bio230143-bucket01/ubna_data_01/recover-20220715/UBNA_008/20220713_000000.WAV')

In [ ]:
activity_dets_arr = actvt.generate_activity_dets_results(data_params, file_paths)
activity_df1 = actvt.construct_activity_grid_for_number_of_dets(activity_dets_arr, data_params["cur_dc_tag"])

In [ ]:

activity_times = pd.DatetimeIndex(activity_df1.index).tz_localize('UTC')
activity_dates = pd.DatetimeIndex(activity_df1.columns).strftime("%m/%d/%y")
activity_times = activity_times.tz_convert(tz='US/Pacific')
ylabel = 'PST'
activity_times = activity_times.strftime("%H:%M")
plot_times = [''] * len(activity_times)
plot_times[::4] = activity_times[::4]
plot_times += ['08:00']
plot_dates = [''] * len(activity_dates)
plot_dates[::7] = activity_dates[::7]
dates_for_sunrise_sunset = pd.to_datetime(activity_df1.columns.values, format='%m/%d/%y')
activity_lat = [SEATTLE_LATITUDE]*len(dates_for_sunrise_sunset)
activity_lon = [SEATTLE_LONGITUDE]*len(dates_for_sunrise_sunset)
sunrise_time = pd.DatetimeIndex(suncalc.get_times(dates_for_sunrise_sunset, activity_lon, activity_lat)['sunrise_end'])
sunset_time = pd.DatetimeIndex(suncalc.get_times(dates_for_sunrise_sunset, activity_lon, activity_lat)['sunset_start'])
sunrise_seconds_from_midnight = sunrise_time.hour * 3600 + sunrise_time.minute*60 + sunrise_time.second
sunset_seconds_from_midnight = sunset_time.hour * 3600 + sunset_time.minute*60 + sunset_time.second

In [12]:
site_key = 'Telephone'
type_key = ''
data_params["site_name"] = SITE_NAMES[site_key]
data_params["site_tag"] = site_key
data_params["type_tag"] = type_key

file_paths = get_file_paths(data_params)

In [13]:
activity_dets_arr = pipeline.run_for_dets(data_params, pipeline_params, file_paths)
activity_df2 = actvt.construct_activity_grid_for_number_of_dets(activity_dets_arr, data_params["cur_dc_tag"])
activity_df3 = activity_df2.reindex(columns=activity_df1.columns, fill_value=np.NaN)

In [14]:
activity_dets_arr

,num_dets (30of30),num_dets (5of10)
datetime_UTC,,
2022-07-23 00:00:00,0.0,0.0
2022-07-23 00:30:00,2.0,0.0
2022-07-23 01:00:00,0.0,0.0
2022-07-23 01:30:00,3.0,3.0
2022-07-23 02:00:00,1.0,0.0
...,...,...
2022-10-17 13:30:00,1136.0,675.0
2022-10-17 14:00:00,5.0,5.0
2022-10-17 14:30:00,0.0,0.0


In [15]:
cmap = plt.get_cmap('viridis')
norm = colors.LogNorm(vmin=1, vmax=10e3)
cmap.set_bad(color='darkred')

plt.rcParams.update({'font.size': (0.8*len(activity_dates) + 0.8*len(activity_times))})
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(1*len(activity_dates), 2*len(activity_times)), sharex=True, sharey=True)
masked_array_for_nodets = np.ma.masked_where(activity_df1.values==np.NaN, activity_df1.values)
im1 = ax1.imshow(1+(masked_array_for_nodets), cmap=cmap, norm=norm)
ax1.plot(np.arange(0, len(plot_dates)), ((sunset_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunset')
ax1.axhline(y=np.where(activity_times=='00:00')[0]-0.5, linewidth=0.5*len(activity_times), linestyle='dashed', color='white', label='Midnight 0:00 PST')
ax1.plot(np.arange(0, len(plot_dates)), ((sunrise_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunrise')
ax1.grid(which='both', linewidth=6, alpha=0.4)
ax1.set_ylabel(f'Time (HH:MM, {ylabel})')
ax1.set_yticks(np.arange(0, len(plot_times))-0.5)
ax1.set_yticklabels(plot_times, ha='right')

masked_array_for_nodets = np.ma.masked_where(activity_df3.values==np.NaN, activity_df3.values)
im2 = ax2.imshow(1+masked_array_for_nodets, cmap=cmap, norm=norm)
ax2.plot(np.arange(0, len(plot_dates)), ((sunset_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunset')
ax2.axhline(y=np.where(activity_times=='00:00')[0]-0.5, linewidth=0.5*len(activity_times), linestyle='dashed', color='white', label='Midnight 0:00 PST')
ax2.plot(np.arange(0, len(plot_dates)), ((sunrise_seconds_from_midnight / (30*60)) % len(plot_times)) - 0.5, 
        color='white', linewidth=0.5*len(activity_times), linestyle='dashed', label=f'Time of Sunrise')
ax2.set_ylabel(f'Time (HH:MM, {ylabel})')
ax2.set_xlabel('Date (MM/DD/YY)')
ax2.grid(which='both', linewidth=6, alpha=0.4)
ax2.set_xticks(np.arange(0, len(activity_df1.columns))-0.5)
ax2.set_xticklabels(plot_dates, rotation=30)
ax2.set_yticks(np.arange(0, len(plot_times)) - 0.5)
ax2.set_yticklabels(plot_times, ha='right')

fig.tight_layout()

pos1 = ax1.get_position(original=False)
pos2 = ax2.get_position(original=False)
colorbar_ax = fig.add_axes([pos2.x1 + 0.02, pos2.y0, 0.02, (pos1.y1 - pos2.y0)])
cbar = fig.colorbar(im1, cax=colorbar_ax, orientation='vertical')
fig.text(x=pos1.x1 - 0.04, y=pos1.y1 + 0.04, s='Number of calls', fontweight='bold',
         fontsize=(1.2*len(activity_dates) + 1.2*len(activity_times)))

plt.show()